# 02 - Speaker-Listener: learning to talk while the other one keeps changing

The **speaker** sees which landmark is the goal but can't move; the **listener** can move but can't see the goal.
They share one reward, so they must invent a little language - *while both are learning at the same time*.

That is **non-stationarity**: from the listener's point of view, the meaning of the speaker's messages keeps changing.

We compare two algorithms that differ in one line - what each agent's critic (its "how good was that?" estimator) sees:

| | critic of agent *i* sees | |
|---|---|---|
| **IDDPG** (independent) | only its own observation and action | the other agent is just part of a changing world |
| **MADDPG** (centralised critic) | everybody's observations and actions | the world it models stays the same while the other learns |

Both act only on their own observation - the difference is only during training. *(Pre-trained runs; nothing trains here except the last cell.)*

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from marl.analysis import (load_run, load_policy, evaluate, video, evolution_frames, plot_curves,
                           plot_matrix, moving_target_matrix, crossplay_matrix, speaker_messages,
                           message_separation, message_drift, pick)
from marl.render import show_video
plt.rcParams["figure.dpi"] = 90

In [ ]:
maddpg, iddpg = load_run("sl_maddpg"), load_run("sl_iddpg")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
plot_curves([maddpg, iddpg], "goal_dist", ax=ax[0], title="listener's final distance to the goal (lower = better)")
plot_curves([maddpg, iddpg], "success", ax=ax[1], title="listener touches the goal (fraction of episodes)")
plt.tight_layout()

## What the agents do, as training goes on
Each clip is the *same* episode replayed with the policy from a later point in training. The text at the bottom is the speaker's message.

In [ ]:
show_video(evolution_frames(maddpg, k=4), fps=10)

In [ ]:
show_video(evolution_frames(iddpg, k=4), fps=10)

## The speaker's "language" over training

The speaker only ever sees the goal colour, so its whole policy is **three messages** (one per possible goal).
* **separation**: how different the messages for different goals are (0 = the listener can't possibly tell goals apart)
* **drift**: how much the messages changed since the previous snapshot - every change is something the listener has to re-learn

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for run in (maddpg, iddpg):
    steps, msgs = speaker_messages(run)
    lab = run["config"]["label"]
    ax[0].plot(steps / 1e6, message_separation(msgs), marker="o", ms=3, label=lab)
    ax[1].plot(steps[1:] / 1e6, message_drift(msgs), marker="o", ms=3, label=lab)
ax[0].set_title("separation between goal messages"); ax[1].set_title("message drift between snapshots")
for a in ax: a.set_xlabel("environment steps (millions)"); a.grid(alpha=.3); a.legend()
plt.tight_layout()

## The moving-target test
Pair the **listener from snapshot *t*** with the **speaker from snapshot *s*** and measure how close the listener gets to the goal.
If the speaker's language kept changing, a listener only works with the speaker it grew up with: good values on the diagonal, bad off it.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, run in zip(ax, (maddpg, iddpg)):
    M, steps = moving_target_matrix(run, k=6, episodes=64)
    plot_matrix(M, steps, ax=a, title=f"{run['config']['label']}: goal distance",
                xlabel="speaker snapshot", ylabel="listener snapshot")
plt.tight_layout()

## Bonus: why not PPO here?
The same task with MAPPO / IPPO (on-policy). Watch the curves: the messages only become useful once the listener reacts to them, and the
listener only learns to react once the messages are useful - on-policy learning never broke out of that chicken-and-egg loop here.
MADDPG's replay buffer and its critic that sees both actions are what make the difference.

In [ ]:
ppo_runs = [load_run(n) for n in ("sl_mappo", "sl_ippo")]
plot_curves([maddpg] + ppo_runs, "goal_dist", title="goal distance: MADDPG vs PPO-family");

## Try it yourself (about 2 minutes on a laptop CPU)
A short MADDPG run from scratch. It won't reach the pre-trained quality, but you can see the curve move. Change `critic="local"` to train IDDPG instead.

In [ ]:
import tempfile, torch
from marl.common.runner import Run
from marl.algos.ddpg import DDPGConfig, train
torch.set_num_threads(2)

steps = 100_000          # ~1-2 min on a laptop CPU
cfg = DDPGConfig(critic="central", warmup_steps=10_000)
run = Run("toy", "speaker_listener", {}, {}, out_root=tempfile.mkdtemp(), eval_episodes=16)
train(run, "speaker_listener", {}, cfg, total_steps=steps, eval_every=20_000, label="my MADDPG")
toy = load_run(run.dir)
plot_curves([toy], "goal_dist", title="my run: goal distance");

In [ ]:
show_video(video(load_policy(toy["ckpts"][-1]), "speaker_listener", {}, title="my MADDPG"))